<div class="lesson-banner">
<span class="lesson-kicker">Python course · 2-hour lesson</span>
<p>Use Python's protocols to stream data, wrap behavior, and manage resources without unnecessary complexity.</p>
</div>

## Learning objectives

- Explain iterable, iterator, and lazy generator behavior.
- Build memory-efficient generator pipelines.
- Write decorators that preserve wrapped-function metadata.
- Use context managers for reliable acquire/release lifecycles.

::: {.callout-note}
### How to use this notebook
Read the explanation, predict each result, run the code, change the inputs, and complete the practice before revealing the solution.
:::


## Iteration is a protocol

An iterable can produce an iterator; an iterator produces values until it raises `StopIteration`. Generator functions use `yield` to suspend and resume state automatically. Laziness reduces memory use and lets downstream consumers stop early, but a generator is normally consumed only once.


In [ ]:
def valid_scores(rows):
    for row in rows:
        score = row.get("score")
        if isinstance(score, (int, float)) and 0 <= score <= 100:
            yield score


rows = [{"score": 82}, {"score": None}, {"score": 95}, {"score": 120}]
scores = valid_scores(rows)
print(next(scores))
print(list(scores))


## Decorators wrap cross-cutting behavior

A decorator accepts a callable and returns a callable. It is useful for timing, authorization, caching, retries, or instrumentation when the policy truly applies across functions. `functools.wraps` preserves the original name and docstring. Avoid decorators that hide important domain flow.


In [ ]:
from functools import wraps
from time import perf_counter


def timed(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        started = perf_counter()
        result = function(*args, **kwargs)
        elapsed_ms = (perf_counter() - started) * 1000
        print(f"{function.__name__} took {elapsed_ms:.3f} ms")
        return result
    return wrapper


@timed
def square_all(values):
    return [value * value for value in values]


print(square_all(range(10)))


## Context managers make cleanup unconditional

A context manager defines what happens on entry and exit. Files, locks, database transactions, and temporary configuration all benefit. `contextlib.contextmanager` is concise for one acquire/yield/release lifecycle; a class is preferable when the lifecycle has richer state.


In [ ]:
from contextlib import contextmanager
from time import perf_counter


@contextmanager
def measured(label: str):
    started = perf_counter()
    try:
        yield
    finally:
        print(f"{label}: {(perf_counter() - started) * 1000:.3f} ms")


with measured("sum squares"):
    result = sum(value * value for value in range(10_000))
print(result)


## Worked example: lazy event pipeline

Each stage accepts and returns an iterable, so the pipeline reads one event at a time and can be tested independently.


In [ ]:
def parse_events(lines):
    for line in lines:
        level, service, message = line.rstrip().split("|", maxsplit=2)
        yield {"level": level, "service": service, "message": message}


def errors_only(events):
    return (event for event in events if event["level"] == "ERROR")


def count_by_service(events):
    counts = {}
    for event in events:
        service = event["service"]
        counts[service] = counts.get(service, 0) + 1
    return counts


lines = [
    "INFO|api|ready",
    "ERROR|model|timeout",
    "ERROR|api|invalid request",
]
print(count_by_service(errors_only(parse_events(lines))))


## Practice lab

Complete these tasks without copying the solution. Test normal, boundary, and invalid inputs where relevant.

1. Write a generator that yields fixed-size batches from any iterable.
2. Create a decorator that counts calls without changing the function result.
3. Create a context manager that temporarily changes a dictionary value and restores it.
4. Explain where laziness helps and where materializing a list is clearer.

::: {.callout-important}
### Practice standard
Your answer should be readable, deterministic, and divided into small functions when the task contains more than one rule.
:::


## Suggested solution

Open the folded code only after attempting every task.


In [ ]:
from contextlib import contextmanager
from functools import wraps


def batched(iterable, size: int):
    if size <= 0:
        raise ValueError("size must be positive")
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch


def count_calls(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return function(*args, **kwargs)
    wrapper.calls = 0
    return wrapper


@contextmanager
def temporary_value(mapping, key, value):
    sentinel = object()
    previous = mapping.get(key, sentinel)
    mapping[key] = value
    try:
        yield mapping
    finally:
        if previous is sentinel:
            mapping.pop(key, None)
        else:
            mapping[key] = previous


print(list(batched(range(7), 3)))


## Knowledge check

**1. What does `yield` change?**

::: {.callout-note collapse="true"}
### Answer
It makes the function return a lazy generator that resumes between values.
:::

**2. Why use `wraps`?**

::: {.callout-note collapse="true"}
### Answer
It preserves metadata of the decorated function.
:::

**3. What does `finally` guarantee in a context manager?**

::: {.callout-note collapse="true"}
### Answer
Cleanup runs whether the body succeeds or fails.
:::


## Recap

- Learn protocols before clever syntax.
- Stream data when early stopping or memory matters.
- Make resource cleanup unconditional.


<div class="lesson-nav">
<a href="09-oop-and-dataclasses.html"><i class="bi bi-arrow-left" aria-hidden="true"></i> Object-Oriented Python and Dataclasses</a>
<a href="11-numpy.html">NumPy for Numerical Computing <i class="bi bi-arrow-right" aria-hidden="true"></i></a>
</div>
